[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/02_%E3%83%95%E3%82%A1%E3%83%8D%E3%83%AB%E3%81%A8%E3%83%81%E3%83%A3%E3%83%8D%E3%83%AB%E5%8A%B9%E7%8E%87.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../ を有効化
    os.makedirs('../data', exist_ok=True)
    import numpy as np, pandas as pd
    D = '../data'; rng = np.random.default_rng(42)
    n = 120; cls = rng.choice(['A','B','C'], n)
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int)
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    days = pd.date_range('2026-06-01', periods=30)
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}
    won = np.array([rng.random()<wr[c] for c in ch])
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}
    rows = []
    for mo in months:
        for c in chs:
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:
        for c in (rng.random(size)<p): ab.append([grp,int(c)])
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 実践マーケ-02. マーケティング・ファネルとチャネル効率

**舞台設定**：半年分のマーケ施策データ `web_marketing.csv` を分析し、
「来年はどのチャネルに予算を回すべきか」を提案します。

**使う統計（4〜3級）**：割合・率の計算・チャネル比較・費用対効果

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False
mk = pd.read_csv('../data/web_marketing.csv')
mk.head()

### 📋 使うデータ：`web_marketing.csv`（月×チャネルのマーケ実績）
1行＝ある月・あるチャネルの実績。表示→クリック→獲得 の数と費用(円)。

| 月 | チャネル | 表示回数 | クリック数 | 獲得数 | 費用 |
|---|---|---|---|---|---|
| 2026-01 | 展示会 | 1855 | 149 | 22 | 4363 |
| 2026-01 | Web広告 | 14854 | 532 | 26 | 39715 |
| 2026-01 | メルマガ | 7258 | 294 | 16 | 1578 |

## 1. マーケティング・ファネル

見込み客は段階的に減っていきます（じょうご＝ファネル）。

```
表示(Impression) → クリック(Click) → 獲得(Conversion=問い合わせ/申込)
```

各段階の「通過率」を計算します。
- **CTR**（クリック率）= クリック ÷ 表示
- **CVR**（獲得率）= 獲得 ÷ クリック

In [ ]:
g = mk.groupby('チャネル')[['表示回数','クリック数','獲得数','費用']].sum()
g['CTR'] = (g['クリック数'] / g['表示回数'] * 100).round(2)
g['CVR'] = (g['獲得数'] / g['クリック数'] * 100).round(2)
g

## 2. 費用対効果（CPA）

**CPA**（Cost Per Acquisition）= 費用 ÷ 獲得数 = 1件獲得するのにかかったお金。
**小さいほど効率が良い**チャネルです。

In [ ]:
g['CPA'] = (g['費用'] / g['獲得数']).round(0)
g[['獲得数','費用','CPA']].sort_values('CPA')

In [ ]:
g['CPA'].sort_values().plot(kind='barh', figsize=(7,4),
    title='チャネル別CPA（低いほど効率的）')
plt.xlabel('1件あたり獲得コスト(円)'); plt.show()

> 💡 **獲得数が多い ≠ 効率が良い**。Web広告は数を稼げてもCPAが高いことが多い。
紹介やメルマガはCPAが低い「おいしい」チャネルになりがちです。

## 3. 効率と量のバランス（バブルチャート）

横軸=獲得数（量）、縦軸=CPA（効率）、円の大きさ=費用 で一望します。
**右下（量が多くCPAが低い）が理想**。

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(g['獲得数'], g['CPA'], s=g['費用']/300, alpha=0.5)
for name, row in g.iterrows():
    ax.annotate(name, (row['獲得数'], row['CPA']))
ax.set_xlabel('獲得数（量）'); ax.set_ylabel('CPA（コスト効率）')
ax.set_title('チャネルの量×効率マップ'); plt.show()

---
## 🏆 ワークシート課題

**課題1.** 最もCVR（獲得率）が高いチャネルと、最もCPAが低いチャネルを答えよう。

**課題2.** 月ごとに「Web広告」のCPAは改善しているか？ 月別に集計して折れ線で確認しよう。

**課題3.**（提案）来年100万円の追加予算がある。どのチャネルにいくら配分するか、
CPAとCVRを根拠に提案文を書こう。

In [ ]:
# 課題1


In [ ]:
# 課題2（ヒント）
ad = mk[mk['チャネル']=='Web広告'].copy()
ad['CPA'] = ad['費用'] / ad['獲得数']


<details>
<summary>▶ 解答例を見る（クリックで開く）</summary>

<pre style="background:#f6f8fa;padding:10px;border-radius:6px;overflow:auto"><code>print(g[&#x27;CVR&#x27;].idxmax(), g[&#x27;CPA&#x27;].idxmin())
ad.groupby(&#x27;月&#x27;)[&#x27;CPA&#x27;].mean().plot(marker=&#x27;o&#x27;); plt.show()</code></pre>
</details>